# IFS ENS — France-Wide ArchiveDownloads IFS ENS ensemble forecasts (51 members, 15 days, 0.25°),processes to daily resolution, and caches at France level.**Storage:** ~200–400 MB per run (51 members × 15 days × 8 variables × France grid).Keep the latest few runs; older ones can be deleted.**Flow:**```ECMWF open data (global GRIB) → open + slice to France → daily aggregation → cache                                                                              ↓                                            66_full_pipeline: extract_parcel_forcing()```

In [ ]:
import osimport loggingfrom datetime import date, timedeltafrom pathlib import PathREPO_ROOT = os.path.dirname(os.getcwd())os.chdir(REPO_ROOT)RAW_DIR = Path("data/raw")PROCESSED_DIR = Path("data/processed")SHIFT = 0  # UTC+0; set to 1 for French timelogging.basicConfig(    level=logging.INFO,    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",    datefmt="%H:%M:%S",)

## Step 1 — Download

In [ ]:
from irrigator.ingestion.ifs_ens_client import (    fetch_ifs_ens,    fetch_latest_ifs_ens,    open_ifs_ens,    process_ifs_ens_to_daily,    save_ifs_daily,    load_ifs_daily,)# Download latest available run (tries today 00Z, yesterday 12Z, yesterday 00Z)grib_path = fetch_latest_ifs_ens(raw_dir=RAW_DIR)print(f"Downloaded: {grib_path}")print(f"Size: {grib_path.stat().st_size / 1e6:.0f} MB")

## Step 2 — Open and slice to France

In [ ]:
# Open GRIB, slice to France bbox (adds ~1° buffer for interpolation)ds_raw = open_ifs_ens(grib_path)print(f"Raw IFS ENS: {dict(ds_raw.sizes)}")print(f"Variables: {list(ds_raw.data_vars)}")

## Step 3 — Process to daily

In [ ]:
# Convert 6-hourly → daily, handle step accumulation for tp/ssrdifs_daily = process_ifs_ens_to_daily(ds_raw, shift_utc=SHIFT)print(f"\nDaily IFS ENS: {dict(ifs_daily.sizes)}")print(f"Variables: {list(ifs_daily.data_vars)}")# Quick sanity checkif "t_mean" in ifs_daily:    t = ifs_daily["t_mean"].isel(number=0)    print(f"T mean range: [{float(t.min()):.1f}, {float(t.max()):.1f}] °C")if "precip_mm" in ifs_daily:    p = ifs_daily["precip_mm"].isel(number=0)    print(f"Precip range: [{float(p.min()):.1f}, {float(p.max()):.1f}] mm/day")

## Step 4 — Save France-level cache

In [ ]:
# Parse run date from filenamestem = grib_path.stem  # e.g. ifs_ens_2026-06-17_00zparts = stem.split("_")run_date = date.fromisoformat(parts[2])run_hour = int(parts[3].replace("z", ""))daily_path = save_ifs_daily(ifs_daily, run_date, run_hour, processed_dir=PROCESSED_DIR)print(f"Cached: {daily_path} ({daily_path.stat().st_size / 1e6:.0f} MB)")

## Step 5 — Extract at parcel level (demo)This is what the 66 pipeline does: load the France-wide cache,then extract per-member forcing at the parcel location.

In [ ]:
from irrigator.config import load_parcel_configfrom irrigator.atmospheric.forcing import extract_parcel_forcing, DailyForcingparcel = load_parcel_config("configs/parcels/example.yaml")# Load cached dailyifs_cached = load_ifs_daily(processed_dir=PROCESSED_DIR)print(f"Loaded: {dict(ifs_cached.sizes)}")# Extract one member to show it worksmember_0 = ifs_cached.sel(number=0) if "number" in ifs_cached.dims else ifs_cached# extract_parcel_forcing expects ERA5-Land variable names — IFS daily has the same# For a quick test without terrain correction:cell = member_0.sel(    longitude=parcel.lon, latitude=parcel.lat, method="nearest",)print(f"\nParcel ({parcel.lat:.2f}°N, {parcel.lon:.2f}°E):")print(f"  Forecast days: {len(cell.valid_time)}")if "t_mean" in cell:    print(f"  T mean: [{float(cell.t_mean.min()):.1f}, {float(cell.t_mean.max()):.1f}] °C")if "precip_mm" in cell:    print(f"  Total precip: {float(cell.precip_mm.sum()):.0f} mm over {len(cell.valid_time)} days")

## Visualise ensemble spread

In [ ]:
import matplotlib.pyplot as pltif "number" in ifs_cached.dims:    cell_all = ifs_cached.sel(        longitude=parcel.lon, latitude=parcel.lat, method="nearest",    )    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)    # Temperature ensemble    ax = axes[0]    for m in cell_all.number.values[:10]:  # plot first 10 members        t = cell_all.sel(number=m)["t_mean"]        ax.plot(t.valid_time.values, t.values, alpha=0.3, color="steelblue")    t_mean = cell_all["t_mean"].mean(dim="number")    ax.plot(t_mean.valid_time.values, t_mean.values, "b-", lw=2, label="Ensemble mean")    ax.set_ylabel("T mean [°C]")    ax.set_title(f"IFS ENS 15-day forecast at ({parcel.lat:.2f}°N, {parcel.lon:.2f}°E)")    ax.legend()    # Precipitation ensemble    ax = axes[1]    p_mean = cell_all["precip_mm"].mean(dim="number")    p_p75 = cell_all["precip_mm"].quantile(0.75, dim="number")    p_p25 = cell_all["precip_mm"].quantile(0.25, dim="number")    ax.fill_between(p_mean.valid_time.values, p_p25.values, p_p75.values,                    alpha=0.3, color="steelblue", label="p25-p75")    ax.bar(p_mean.valid_time.values, p_mean.values, alpha=0.7, color="steelblue", label="mean")    ax.set_ylabel("Precip [mm/day]")    ax.legend()    plt.tight_layout()    plt.show()